# City of Boston Public Notices - RAG Agent
Reads the `chroma_db/` built by `notice-scraping.ipynb`. (Read only)

Connect to ChromaDB

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# should match notice-scraping.ipynb
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "public_notices"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DB_PATH,
)

c:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18347.63it/s]


Checking whats in the collection

In [2]:
from collections import Counter

print("chunks:", vectorstore._collection.count())

all_meta = vectorstore.get(include=["metadatas"])["metadatas"]

# field names and types, so we catch schema changes early
print("\nmetadata schema:")
for k, v in sorted(all_meta[0].items()):
    print(f"  {k}: {type(v).__name__} = {str(v)[:60]}")

# collapse to one entry per notice, since metadata repeats across chunks
by_notice = {m["notice_id"]: m for m in all_meta}
print("\nnotices:", len(by_notice))
print("source_type:", Counter(m.get("source_type") for m in all_meta))
print("public_testimony:", Counter(m.get("public_testimony") for m in by_notice.values()))
print("cancelled:", Counter(m.get("cancelled") for m in by_notice.values()))

posted = sorted(m.get("posted_at", "") for m in by_notice.values())
print("posted_at range:", posted[0][:10], "to", posted[-1][:10])

chunks: 506

metadata schema:
  address_1: str = Boston City Hall 
  address_2: str = 1 City Hall Square, Board Room, Room 816
  cancelled: bool = False
  checked_at: str = 2026-07-30T20:08:57.105491+00:00
  event_datetime: str = 2026-08-19T13:00:00Z
  file_hash: str = b75ee84e481db0befebdcfffa2bff88e6e003bc0cd64940ac24b3cc8acf6
  file_label: str = Official Filed Posting
  notice_id: str = 16492916
  notice_url: str = https://www.boston.gov/public-notices/16492916
  posted_at: str = 2025-11-19T10:59:00-05:00
  public_testimony: bool = False
  source_type: str = pdf
  status: str = ok
  title: str = Boston Retirement Board Meeting | Boston.gov

notices: 60
source_type: Counter({'pdf': 325, 'page_text': 181})
public_testimony: Counter({False: 36, True: 24})
cancelled: Counter({False: 59, True: 1})
posted_at range: 2025-11-19 to 2026-07-30


Retrieval test (lower score is better)

In [3]:
for q in ["when is the retirement board meeting?",
          "how do I submit public comment?",
          "what notices are about tree removal?",
          "which meetings are happening in August?"]:
    print("\n#####", q)
    for doc, score in vectorstore.similarity_search_with_score(q, k=3):
        m = doc.metadata
        title = (m.get("title") or "").replace(" | Boston.gov", "")
        print(f"  [{score:.3f}] {m.get('source_type')} | {title}")
        print(f"        {doc.page_content[:110]}")


##### when is the retirement board meeting?
  [0.537] pdf | Boston Retirement Board Meeting
        Boston Retirement System
Sean F. Kelly Karen T. Cross
Scott M. Finn
EXECUTIVE OFFICER
Timothy J. Smyth, Esquir
  [0.537] pdf | Boston Retirement Board Meeting
        Boston Retirement System
Sean F. Kelly Karen T. Cross
Scott M. Finn
EXECUTIVE OFFICER
Timothy J. Smyth, Esquir
  [0.537] pdf | Boston Retirement Board Meeting
        Boston Retirement System
Sean F. Kelly Karen T. Cross
Scott M. Finn
EXECUTIVE OFFICER
Timothy J. Smyth, Esquir

##### how do I submit public comment?
  [1.346] page_text | City Council Committee on Government Operations Hearing on Docket #0998
        Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to tes
  [1.346] page_text | City Council Committee on Ways and Means Hearing on Docket #0586
        Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to 

Metadata filtering: same scores, smaller pool

In [4]:
query = "how do I submit public comment?"

# Two separate searches, one per source_type. Scores are unchanged from the unfiltered run, only the set of eligible chunks differs.
for source_type in ["page_text", "pdf"]:
    print(f"\n### {source_type}")
    for doc, score in vectorstore.similarity_search_with_score(
        query, k=2, filter={"source_type": source_type}
    ):
        print(f"  [{score:.3f}] {doc.page_content[:130]}")


### page_text
  [1.346] Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to testify in person shoul
  [1.346] Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to testify in person shoul

### pdf
  [1.436] PUBLIC HEARING AGENDA
DATE:     Wednesday, August 12, 2026
TIME:     4:30 - 6:30 PM
PLACE: This meeting will be held virtually onl
  [1.475] will hold a public
APP # 26.1070 SE


OpenAI client

In [5]:
import os
import json
from openai import OpenAI

with open("open_ai_api_key.txt", encoding="utf-8-sig") as f:
    key = f.read().strip().strip('"').strip("'")

if not key.startswith("sk-"):
    raise ValueError(f"Key doesn't look valid: {len(key)} chars starting {key[:6]!r}")

os.environ["OPENAI_API_KEY"] = key
client = OpenAI()

LLM_MODEL = "gpt-4o-mini"

Query planning: split the question into search text + metadata filters

In [6]:
# fields that actually exist in our metadata.
ALLOWED_FILTERS = {"source_type", "cancelled", "notice_id", "public_testimony"}

EXTRACT_PROMPT = """You convert a user's question about Boston public notices into a search plan.

Available metadata fields for filtering:
- source_type: "pdf" or "page_text"
- cancelled: true or false
- public_testimony: true or false, whether the public can testify
- notice_id: a string of digits, e.g. "16492916"

Return ONLY valid JSON, no markdown fences, in this shape:
{{"search_text": "<the topical part of the question to search semantically>",
  "filters": {{"<field>": <value>}}}}

Use "filters" only for constraints that map to the fields listed above.
Remove from search_text any wording that you converted into a filter.
Anything about dates, times, or neighborhoods should stay in search_text for now.
If there are no applicable filters, use an empty object.

Question: {question}"""


def extract_search_plan(question):
    """Returns (search_text, filters)."""
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": EXTRACT_PROMPT.format(question=question)}],
        response_format={"type": "json_object"},   
    )
    plan = json.loads(resp.choices[0].message.content)

    # drop filters on fields we dont store
    raw_filters = plan.get("filters") or {}
    filters = {k: v for k, v in raw_filters.items() if k in ALLOWED_FILTERS}
    dropped = set(raw_filters) - set(filters)
    if dropped:
        print("  (dropped unsupported filters:", dropped, ")")

    return plan.get("search_text", question), filters or None

In [7]:
for q in ["how do I submit public comment?",
          "which notices allow public testimony?",
          "are there any cancelled notices?",
          "what meetings are happening at 4 pm?"]:
    search_text, filters = extract_search_plan(q)
    print(f"\nQ: {q}\n  search_text: {search_text!r}\n  filters: {filters}")


Q: how do I submit public comment?
  search_text: 'submit public comment in Boston'
  filters: None

Q: which notices allow public testimony?
  search_text: 'notices allow public testimony'
  filters: {'public_testimony': True}

Q: are there any cancelled notices?
  search_text: 'are there any notices'
  filters: {'cancelled': True}

Q: what meetings are happening at 4 pm?
  search_text: 'meetings happening at 4 pm'
  filters: None


RAG pipeline: 

In [8]:
def answer(question, k=4):
    """Returns (answer_text, hits). hits are in the same order as the [1..k] citations."""
    # 1. planning
    search_text, filters = extract_search_plan(question)

    # 2. retrieving
    hits = vectorstore.similarity_search(search_text, k=k, filter=filters)

    # 3. filter matching nothing 
    if not hits:
        return (
            "I couldn't find any public notices matching that. "
            f"(searched for {search_text!r} with filters {filters})"
        ), []

    # 4. numbering chunks so model can cite them
    context = "\n\n".join(
        f"[{i + 1}] {(d.metadata.get('title') or '').replace(' | Boston.gov', '')}"
        f" (notice {d.metadata.get('notice_id')}, event {d.metadata.get('event_datetime')}, "
        f"{d.metadata.get('source_type')})\n{d.page_content}"
        for i, d in enumerate(hits)
    )

    # 5. generating 
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say you don't know.
Cite the source number after each individual claim, not once at the end.
If a claim draws on multiple sources, cite all of them.

Each numbered block belongs to a specific notice. Never combine details from
different notices into one statement. If several notices match, list them
separately with their dates.

Context:
{context}

Question: {question}"""

    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content, hits


def print_sources(hits):
    if not hits:
        return
    print("\nSources:")
    for i, h in enumerate(hits, 1):
        m = h.metadata
        title = (m.get("title") or "").replace(" | Boston.gov", "")
        where = m.get("file_label") or "notice webpage"
        url = m.get("notice_url") or m.get("detail_url")
        print(f" [{i}] [{m.get('source_type')}] {where} - {title}")
        print(f"     {url}")

End to end

In [9]:
for q in ["how do I submit public comment?",              # normal question
          "are there any cancelled notices?",             # filter with exactly one match
          "when is the retirement board meeting?"]:       # several near-identical notices
    print("\n" + "=" * 70)
    print("Q:", q)
    ans, hits = answer(q)
    print(ans)
    print_sources(hits)


Q: how do I submit public comment?
To submit public comment for the different events, you can follow these instructions:

1. For the City Council Committee on Government Operations Hearing (notice 16600991), you can send written comments to the Committee email (ccc.go@boston.gov) or the staff email (meghan.kavanagh@boston.gov). Public testimony can also be shared virtually via videoconference by emailing the staff contact for a link and instructions [1].

2. For the Boston Conservation Commission Public Hearing (notice 16602081), public comments can be shared during the hearing, and written testimony can be accepted via email at cc@boston.gov prior to the hearing until Wednesday, August 5, 2026, at 3:30 pm [3].

3. For the Office of Participatory Budgeting External Oversight Board Meeting (notice 16596806), public comment will be part of the agenda during the meeting, but specific details on how to submit or participate are not provided in the context [2].

4. For the Boston Civic Des

checking can we filter on date ranges?

In [10]:
# Does Chroma support range comparisons on ISO date strings?
# If not, date filtering needs a numeric field from the pipeline.
try:
    r = vectorstore.get(where={"posted_at": {"$gte": "2026-07-01"}}, limit=5)
    print("string range works, matched:", len(r["ids"]))
except Exception as e:
    print("string range NOT supported:", type(e).__name__, e)

string range NOT supported: ValueError Expected operand value to be an int or a float for operator $gte, got 2026-07-01 in get.
